In [1]:
!pip install faker

In [2]:
import os
import pandas as pd
import random

from faker import Faker

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
# Configurações de semente para reprodutibilidade
random.seed(42)
fake = Faker()

In [5]:
directory = "/content/drive/My Drive/KOA RecomSys/"

In [14]:
# Configurações do Dataset
NUM_USERS = 1200
NUM_PRODUCTS = 3500
NUM_BRANDS = 300
NUM_CATEGORIES = 200
TOTAL_RELATIONS = 18600

In [15]:
# 1. Gerar Entidades com Descrições
user_data = []
for i in range(NUM_USERS):
    user_data.append({
        "id": f"U{i:04d}",
        "name": fake.name(),
        "profile": random.choice(["Gamer", "Software Engineer", "Designer", "SRE Admin", "Office Worker", "Student"])
    })

brand_data = []
brand_names = ["Logitech", "Samsung", "TP-Link", "Apple", "Dell", "HP", "Sony", "Microsoft", "Asus", "Crucial"]
for i in range(NUM_BRANDS):
    b_name = random.choice(brand_names) + f" {fake.company_suffix()}" if i >= len(brand_names) else brand_names[i]
    brand_data.append({
        "id": f"B{i:03d}",
        "name": b_name,
        "description": f"Leading manufacturer specializing in {fake.bs()}."
    })

product_categories = [
    "Peripherals", "Storage", "Networking", "Monitors", "Laptops",
    "Accessories", "Audio", "Smart Home", "Components", "Cables"
]
category_data = [{"id": f"C{i:03d}", "name": random.choice(product_categories)} for i in range(NUM_CATEGORIES)]

product_data = []
tech_adjectives = ["Wireless", "High-Speed", "Ergonomic", "Mechanical", "Ultrawide", "Portable", "Gaming", "Cloud-Ready"]
for i in range(NUM_PRODUCTS):
    adj = random.choice(tech_adjectives)
    cat = random.choice(category_data)
    brand = random.choice(brand_data)
    p_name = f"{brand['name']} {adj} {cat['name']} Model-{random.randint(100, 999)}"

    product_data.append({
        "id": f"P{i:04d}",
        "name": p_name,
        "brand_id": brand['id'],
        "category_id": cat['id'],
        "description": f"A {adj.lower()} {cat['name'].lower()} designed for professional use, featuring {fake.catch_phrase().lower()}."
    })

In [16]:
# 2. Gerar Relações (Triplas: Head, Relation, Tail)
triples = []

# Mapeamento Estrutural (Atributos como relações)
for p in product_data:
    triples.append([p['id'], "belongs_to_brand", p['brand_id']])
    triples.append([p['id'], "belongs_to_category", p['category_id']])

# Compras (buys) - Usuário -> Produto
for _ in range(5000):
    triples.append([random.choice(user_data)['id'], "buys", random.choice(product_data)['id']])

# Adicionando 1.500 relações de menção para enriquecer o perfil do usuário
for _ in range(1500):
    triples.append([random.choice(user_data)['id'], "mentions", random.choice(category_data)['id']])

# Relações de Recomendação (also_buy, also_view)
for _ in range(2500):
    p1, p2 = random.sample(product_data, 2)
    triples.append([p1['id'], "also_buy", p2['id']])

while len(triples) < TOTAL_RELATIONS:
    p1, p2 = random.sample(product_data, 2)
    triples.append([p1['id'], "also_view", p2['id']])

In [17]:
# 3. Exportação
df_triples = pd.DataFrame(triples, columns=['head', 'relation', 'tail'])
df_entities = pd.concat([
    pd.DataFrame(user_data),
    pd.DataFrame(brand_data),
    pd.DataFrame(product_data),
    pd.DataFrame(category_data)
], ignore_index=True)

df_triples.to_csv("triples.csv", index=False)
df_entities.to_csv("entities_metadata.csv", index=False)

In [18]:
triples_path = os.path.join(directory, "triples.csv")
entities_path = os.path.join(directory, "entities_metadata.csv")

df_triples.to_csv(triples_path, index=False)
df_entities.to_csv(entities_path, index=False)

print(f"✅ Success! Files saved in: {directory}")
print(f"Total Triples: {len(df_triples)}")
print(f"Total Entities: {len(df_entities)}")

✅ Success! Files saved in: /content/drive/My Drive/KOA RecomSys/
Total Triples: 18600
Total Entities: 5200
